# LLM Quantization Toolkit

This notebook implements and compares multiple quantization methods for Large Language Models:

- **RTN (Round-To-Nearest)**: Simple weight-only quantization with group-wise scaling
- **GPTQ**: Advanced quantization with sequential layer-wise optimization  
- **AWQ (Activation-aware Weight Quantization)**: Preserves important weights based on activation magnitudes

## Target Model
- **Model**: OpenLLaMA 7B
- **Output**: Quantized models at 4-bit and 8-bit precision
- **Calibration**: WikiText-2 dataset (128 samples)

## Use Cases
- Mobile deployment (reduced memory footprint)
- Edge devices (lower bandwidth requirements)
- Cost-effective inference (reduced compute)

## 1. Imports and Configuration

In [1]:
# ============================================================================
# IMPORTS AND CONFIGURATION
# ============================================================================

# Standard library imports
from pathlib import Path
from typing import Iterable, List

# PyTorch and deep learning
import torch
import torch.nn as nn

# Hugging Face transformers and datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

# Project imports
import utils.model_loader as model_loader
import utils.dataloader as dataloader

# ============================================================================
# CONFIGURATION
# ============================================================================

# Model configuration
MODEL_ID = "openlm-research/open_llama_7b"  # Target model to quantize

# Output configuration  
OUTPUT_ROOT = Path("./quantized_models")  # Directory for saving quantized models

# Calibration configuration
MIN_CALIB_SAMPLES = 128  # Minimum number of calibration samples for quantization

In [2]:
# Check if cache is on the mounted drive
from huggingface_hub.constants import HF_HUB_CACHE
print(f"Hugging Face Hub is pointing to: {HF_HUB_CACHE}")

Hugging Face Hub is pointing to: /mnt/huggingface_cache/hub


## 2. Calibration Data Loading

Calibration data is crucial for quantization algorithms (especially GPTQ and AWQ) to determine optimal quantization parameters. We use WikiText-2, a standard language modeling dataset with diverse, high-quality text.

In [3]:
# Load calibration data (this runs when the cell executes)
CALIB_TEXTS = dataloader.load_wikitext_calibration(num_samples=MIN_CALIB_SAMPLES)

Loading WikiText-2 dataset for calibration...


✓ Loaded 128 calibration samples from WikiText-2


## 3. RTN (Round-To-Nearest) Quantization

**RTN** is the simplest quantization method:
- **Approach**: Directly round weights to nearest quantized value
- **Speed**: Very fast (no optimization required)
- **Accuracy**: Lower than GPTQ/AWQ but still usable
- **Use case**: Quick quantization when speed matters more than accuracy

**Features**:
- Group-wise quantization for better accuracy
- Per-channel scaling
- Weight-only quantization (activations remain in FP16)

In [4]:
# ============================================================================
# RTN (ROUND-TO-NEAREST) QUANTIZATION IMPLEMENTATION
# ============================================================================

def _rtn_quantize_tensor(w: torch.Tensor, bits: int, group_size: int = 128) -> torch.Tensor:
    """
    Quantize a weight tensor using Round-To-Nearest method with group-wise scaling.
    
    Args:
        w: Weight tensor to quantize
        bits: Target bit width (e.g., 4 for 4-bit, 8 for 8-bit)
        group_size: Size of groups for group-wise quantization (smaller = more accurate but larger overhead)
    
    Returns:
        Quantized and dequantized weight tensor (still in original dtype)
    """
    # Skip quantization for 16-bit or higher
    if bits >= 16:
        return w

    # Calculate quantization range for signed integers
    qmin = -(2 ** (bits - 1))  # e.g., -8 for 4-bit
    qmax = (2 ** (bits - 1)) - 1  # e.g., 7 for 4-bit

    orig_shape = w.shape
    w_flat = w.reshape(-1, w.shape[-1])  # Flatten to 2D for processing

    # Per-channel quantization (when group_size is too large or disabled)
    if group_size <= 0 or group_size >= w_flat.shape[1]:
        # Find max absolute value per channel
        max_abs = w_flat.abs().amax(dim=1, keepdim=True).clamp(min=1e-8)
        scale = max_abs / qmax
        
        # Quantize and dequantize
        q = torch.round(w_flat / scale).clamp(qmin, qmax)
        w_q = q * scale
    
    # Group-wise quantization (more accurate)
    else:
        n_cols = w_flat.shape[1]
        
        # Pad if necessary to make columns divisible by group_size
        pad = (group_size - (n_cols % group_size)) % group_size
        if pad > 0:
            w_flat = torch.cat([
                w_flat, 
                torch.zeros(w_flat.size(0), pad, device=w.device, dtype=w.dtype)
            ], dim=1)

        # Reshape into groups
        w_grp = w_flat.view(w_flat.size(0), -1, group_size)
        
        # Find max absolute value per group
        max_abs = w_grp.abs().amax(dim=2, keepdim=True).clamp(min=1e-8)
        scale = max_abs / qmax
        
        # Quantize and dequantize
        q = torch.round(w_grp / scale).clamp(qmin, qmax)
        w_q = (q * scale).view(w_flat.size(0), -1)

        # Remove padding
        if pad > 0:
            w_q = w_q[:, :n_cols]

    return w_q.reshape(orig_shape).to(w.dtype)


@torch.no_grad()
def quantize_rtn_inplace(model: nn.Module, bits: int, group_size: int = 128):
    """
    Apply RTN quantization to all Linear layers in the model (in-place).
    
    Args:
        model: Model to quantize
        bits: Target bit width
        group_size: Group size for quantization
    
    Returns:
        Modified model (also modifies in-place)
    """
    for module in model.modules():
        if isinstance(module, nn.Linear):
            # Quantize the weight matrix
            module.weight.data = _rtn_quantize_tensor(
                module.weight.data, 
                bits=bits, 
                group_size=group_size
            )
    return model


def quantize_and_save_rtn(
    model_id: str,
    bits_list: Iterable[int],
    out_root: Path = OUTPUT_ROOT,
    group_size: int = 128,
):
    """
    Quantize model using RTN at multiple bit widths and save results.
    
    Args:
        model_id: HuggingFace model ID or path
        bits_list: List of bit widths to quantize to (e.g., [4, 8])
        out_root: Root directory for saving quantized models
        group_size: Group size for quantization
    """
    # Load tokenizer once (shared across all bit widths)
    tok = model_loader.load_tokenizer(model_id)
    
    # Quantize at each bit width
    for bits in bits_list:
        print(f"\n{'='*60}")
        print(f"Quantizing with RTN at {bits}-bit...")
        print(f"{'='*60}")
        
        # Load fresh model for each quantization
        model = model_loader.load_base_model(model_id)
        
        # Apply RTN quantization
        quantize_rtn_inplace(model, bits=bits, group_size=group_size)

        # Save quantized model
        out_dir = out_root / f"rtn_w{bits}"
        out_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(out_dir)
        tok.save_pretrained(out_dir)
        
        print(f"✓ Saved RTN {bits}-bit model to {out_dir}")
        
        # Clean up
        del model
        torch.cuda.empty_cache()

## 5. GPTQ (Generative Pre-trained Transformer Quantization)

**GPTQ** is an advanced quantization method that optimizes quantization parameters:
- **Approach**: Sequential layer-wise quantization with Hessian-based optimization
- **Speed**: Slower than RTN (requires calibration and optimization)
- **Accuracy**: Much better than RTN, minimal performance loss
- **Use case**: Production deployments where accuracy is critical

**How it works**:
1. For each layer, compute Hessian matrix (sensitivity to weight changes)
2. Quantize weights sequentially, minimizing reconstruction error
3. Use calibration data to guide the optimization

**Reference**: [GPTQ Paper](https://arxiv.org/abs/2210.17323)

In [5]:
# ============================================================================
# GPTQ (GENERATIVE PRE-TRAINED TRANSFORMER QUANTIZATION) IMPLEMENTATION
# ============================================================================

def _build_gptq_examples(tokenizer, texts: List[str], max_len: int = 256):
    """
    Prepare calibration examples for GPTQ quantization.
    
    Args:
        tokenizer: Tokenizer instance
        texts: List of calibration text strings
        max_len: Maximum sequence length
    
    Returns:
        List of tokenized examples (dicts with input_ids and attention_mask)
    """
    examples = []
    for t in texts:
        # Tokenize each text
        enc = tokenizer(t, return_tensors="pt", truncation=True, max_length=max_len)
        examples.append({
            "input_ids": enc["input_ids"][0],
            "attention_mask": enc["attention_mask"][0],
        })
    return examples

# autoq version
def quantize_and_save_gptq(
    model_id: str,
    bits_list: Iterable[int],
    out_root: Path = OUTPUT_ROOT,
    calib_texts: List[str] = None,
    group_size: int = 128,
):
    """
    Quantize model using GPTQ at multiple bit widths and save results.
    
    GPTQ requires calibration data and performs layer-wise optimization
    to minimize quantization error.
    
    Args:
        model_id: HuggingFace model ID or path
        bits_list: List of bit widths to quantize to (e.g., [4])
        out_root: Root directory for saving quantized models
        calib_texts: Calibration texts (uses CALIB_TEXTS if None)
        group_size: Group size for quantization
    """
    from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig

    # Use default calibration texts if not provided
    calib_texts = calib_texts or CALIB_TEXTS
    
    # Load tokenizer and prepare examples
    tok = load_tokenizer(model_id)
    examples = _build_gptq_examples(tok, calib_texts)

    # Quantize at each bit width
    for bits in bits_list:
        print(f"\n{'='*60}")
        print(f"Quantizing with GPTQ at {bits}-bit...")
        print(f"This may take several minutes...")
        print(f"{'='*60}")
        
        # Configure GPTQ quantization
        quantize_config = BaseQuantizeConfig(
            bits=bits,  # Target bit width
            group_size=group_size,  # Group size for quantization
            desc_act=False,  # Disable activation order optimization (faster)
        )

        # Load model with GPTQ wrapper
        model = AutoGPTQForCausalLM.from_pretrained(
            model_id,
            quantize_config=quantize_config,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto",
        )

        # Perform GPTQ quantization (this is where optimization happens)
        print("Running GPTQ optimization...")
        model.quantize(examples)

        # Save quantized model
        out_dir = out_root / f"gptq_w{bits}"
        out_dir.mkdir(parents=True, exist_ok=True)
        model.save_quantized(out_dir, use_safetensors=True)
        tok.save_pretrained(out_dir)
        
        print(f"✓ Saved GPTQ {bits}-bit model to {out_dir}")

        # Clean up
        del model
        torch.cuda.empty_cache()

# # optimum
# def quantize_and_save_gptq(
#     model_id: str,
#     bits_list: Iterable[int],
#     out_root: Path = Path("./output"),
#     calib_texts: List[str] = None,
#     group_size: int = 128,
# ):
#     """
#     Quantize model using Optimum's GPTQ wrapper.
#     """
#     from optimum.gptq import GPTQQuantizer
#     from optimum.gptq.data_config import QuantizeConfig

#     # Use default calibration texts if not provided
#     calib_texts = calib_texts or CALIB_TEXTS
    
#     # Load tokenizer
#     tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
#     if tok.pad_token is None:
#         tok.pad_token = tok.eos_token
        
#     examples = _build_gptq_examples(tok, calib_texts)

#     for bits in bits_list:
#         print(f"\n{'='*60}")
#         print(f"Quantizing with OPTIMUM GPTQ at {bits}-bit...")
#         print(f"{'='*60}")
        
#         # 1. Initialize the Quantizer
#         quantizer = GPTQQuantizer(
#             bits=bits, 
#             dataset=examples, # You can pass your list of dicts directly here
#             model_seqlen=256, 
#             block_name_to_quantize=None, # Automatically inferred for most models
#             desc_act=False,
#             sym=True,
#             group_size=group_size
#         )

#         # 2. Load the base model in FP16
#         model = AutoModelForCausalLM.from_pretrained(
#             model_id,
#             torch_dtype=torch.float16,
#             device_map="auto",
#             low_cpu_mem_usage=True
#         )

#         # 3. Perform Quantization
#         # Optimum handles the layer-wise optimization internally
#         print("Running GPTQ optimization via Optimum...")
#         quantized_model = quantizer.quantize_model(model, tok)

#         # 4. Save the results
#         out_dir = out_root / f"gptq_w{bits}"
#         out_dir.mkdir(parents=True, exist_ok=True)
        
#         # Note: save_pretrained works for the model; quantizer helps save the config
#         quantized_model.save_pretrained(out_dir, use_safetensors=True)
#         tok.save_pretrained(out_dir)
        
#         print(f"✓ Saved GPTQ {bits}-bit model to {out_dir}")

#         # Clean up
#         del model
#         del quantized_model
#         torch.cuda.empty_cache()

## 6. AWQ (Activation-aware Weight Quantization)

**AWQ** protects important weights based on activation magnitudes:
- **Approach**: Identifies and protects salient (important) weights that correspond to large activations
- **Speed**: Moderate (faster than GPTQ, slower than RTN)
- **Accuracy**: Comparable to or better than GPTQ, especially at 4-bit
- **Use case**: Best balance of speed and accuracy for 4-bit quantization

**How it works**:
1. Analyze activation magnitudes on calibration data
2. Identify "salient" weights (those activated strongly)
3. Apply per-channel scaling to preserve these weights
4. Quantize with protection for important weights

**Key advantage**: Better preserves model performance at very low bit widths (3-4 bit)

**Reference**: [AWQ Paper](https://arxiv.org/abs/2306.00978)

In [6]:
# ============================================================================
# AWQ (ACTIVATION-AWARE WEIGHT QUANTIZATION) IMPLEMENTATION
# ============================================================================

def quantize_and_save_awq(
    model_id: str,
    bits_list: Iterable[int],
    out_root: Path = OUTPUT_ROOT,
    calib_texts: List[str] = None,
    group_size: int = 128,
    max_calib_seq_len: int = 256,
):
    """
    Quantize model using AWQ at multiple bit widths and save results.
    
    AWQ analyzes activation patterns to identify and protect important weights,
    resulting in better accuracy at low bit widths.
    
    Args:
        model_id: HuggingFace model ID or path
        bits_list: List of bit widths (AWQ typically supports 4-bit)
        out_root: Root directory for saving quantized models
        calib_texts: Calibration texts (uses CALIB_TEXTS if None)
        group_size: Group size for quantization
        max_calib_seq_len: Maximum sequence length for calibration
    """
    from awq import AutoAWQForCausalLM

    # Use default calibration texts if not provided
    calib_texts = calib_texts or CALIB_TEXTS
    
    # Load tokenizer
    tok = load_tokenizer(model_id)

    # Quantize at each bit width
    for bits in bits_list:
        print(f"\n{'='*60}")
        print(f"Quantizing with AWQ at {bits}-bit...")
        print(f"This may take several minutes...")
        print(f"{'='*60}")
        
        # AWQ primarily supports 4-bit quantization
        if bits not in (4,):
            print(f"Warning: AWQ commonly supports 4-bit. {bits}-bit may not work.")
            raise ValueError("AutoAWQ commonly supports 4-bit weight quantization.")

        # Load model with AWQ wrapper
        model = AutoAWQForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto",
        )

        # Configure AWQ quantization
        quant_config = {
            "w_bit": bits,  # Weight bit width
            "q_group_size": group_size,  # Group size
            "zero_point": True,  # Use zero-point quantization
            "version": "GEMM",  # Use GEMM-optimized kernels
        }

        # Perform AWQ quantization (analyzes activations and quantizes)
        print("Running AWQ optimization...")
        model.quantize(
            tok,
            quant_config=quant_config,
            calib_data=calib_texts,  # Calibration texts (not tokenized)
            max_calib_samples=len(calib_texts),
            max_calib_seq_len=max_calib_seq_len,
        )

        # Save quantized model
        out_dir = out_root / f"awq_w{bits}"
        out_dir.mkdir(parents=True, exist_ok=True)
        model.save_quantized(out_dir)
        tok.save_pretrained(out_dir)
        
        print(f"✓ Saved AWQ {bits}-bit model to {out_dir}")

        # Clean up
        del model
        torch.cuda.empty_cache()

## 7. Run Quantization

Execute all quantization methods and save results.

**⚠️ IMPORTANT**: This cell will:
- Download the 7B parameter model (~13 GB)
- Quantize it using RTN, GPTQ, and AWQ
- Take significant time (30-60 minutes depending on hardware)
- Require ~24GB GPU memory

**Output**: Quantized models will be saved to `./quantized_models/`

In [7]:
# ============================================================================
# EXECUTE QUANTIZATION
# ============================================================================

if __name__ == "__main__":
    # Create output directory
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    
    print("\n" + "="*70)
    print("STARTING QUANTIZATION PIPELINE")
    print("="*70)
    print(f"Model: {MODEL_ID}")
    print(f"Output directory: {OUTPUT_ROOT}")
    print(f"Calibration samples: {len(CALIB_TEXTS)}")
    print("="*70 + "\n")

    # RTN Quantization (Fast, multiple bit widths)
    print("\n📊 STEP 1/3: RTN Quantization")
    quantize_and_save_rtn(
        MODEL_ID, 
        bits_list=[4, 8],  # Quantize to both 4-bit and 8-bit
        out_root=OUTPUT_ROOT
    )


STARTING QUANTIZATION PIPELINE
Model: openlm-research/open_llama_7b
Output directory: quantized_models
Calibration samples: 128


📊 STEP 1/3: RTN Quantization


ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

In [9]:
# GPTQ Quantization (High accuracy, slower)
print("\n📊 STEP 2/3: GPTQ Quantization")
quantize_and_save_gptq(
    MODEL_ID, 
    bits_list=[4],  # 4-bit GPTQ
    out_root=OUTPUT_ROOT
)


📊 STEP 2/3: GPTQ Quantization


/mnt/miniconda3/envs/quant-env/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/mnt/miniconda3/envs/quant-env/lib/python3.10/site-packages/auto_gptq/nn_modules/triton_utils/kernels.py:360: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, input, qweight, scales, qzeros, g_idx, bits, maxq):
/mnt/miniconda3/envs/quant-env/lib/python3.10/site-packages/auto_gptq/nn_modules/triton_utils/kernels.py:368: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/mnt/miniconda3/envs/quant-env/lib/python3.10/site-packages/auto_gptq/nn_modules/triton_utils/kernels.py:399: FutureWarn


Quantizing with GPTQ at 4-bit...
This may take several minutes...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Running GPTQ optimization...


CUDA extension not installed.


✓ Saved GPTQ 4-bit model to quantized_models/gptq_w4


In [21]:
# AWQ Quantization (Activation-aware)
print("\n📊 STEP 3/3: AWQ Quantization")
quantize_and_save_awq(
    MODEL_ID, 
    bits_list=[4],  # 4-bit AWQ
    out_root=OUTPUT_ROOT
)

print("\n" + "="*70)
print("✓ QUANTIZATION COMPLETE!")
print("="*70)
print(f"\nQuantized models saved to: {OUTPUT_ROOT}")
print("\nGenerated models:")
print("  - rtn_w4/  : 4-bit RTN quantized")
print("  - rtn_w8/  : 8-bit RTN quantized")
print("  - gptq_w4/ : 4-bit GPTQ quantized")
print("  - awq_w4/  : 4-bit AWQ quantized")
print("\n" + "="*70 + "\n")


📊 STEP 3/3: AWQ Quantization


/mnt/miniconda3/envs/quant-env/lib/python3.10/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)


ImportError: cannot import name 'BaseImageProcessor' from 'transformers' (/mnt/miniconda3/envs/quant-env/lib/python3.10/site-packages/transformers/__init__.py)

---

# Part 2: Evaluation and Analysis

Once you have quantized models, proceed to the evaluation section below to:
- Compare quantization methods
- Analyze module sensitivity  
- Identify optimal trade-offs for deployment

---

# Model Evaluation and Analysis

This section provides comprehensive evaluation of base and quantized models including:
- **Perplexity** on WikiText-2 test set
- **Downstream task performance** using LM Evaluation Harness (coding, math, reasoning)
- **Module sensitivity analysis** (MLP vs Attention layers)
- **Trade-off visualization** (accuracy vs memory bandwidth)

In [11]:
# Install required dependencies for evaluation
# Run this cell first if you haven't installed these packages

import sys
import subprocess

def install_if_needed(package_name, import_name=None):
    """Install package if not already installed."""
    if import_name is None:
        import_name = package_name
    
    try:
        __import__(import_name)
        print(f"✓ {package_name} already installed")
    except ImportError:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name, "-q"])
        print(f"✓ {package_name} installed successfully")

# Core packages (should already be installed)
install_if_needed("torch")
install_if_needed("transformers")
install_if_needed("datasets")
install_if_needed("accelerate")

# Quantization libraries
install_if_needed("auto-gptq", "auto_gptq")
install_if_needed("autoawq", "awq")

# Evaluation packages
install_if_needed("lm-eval[api]", "lm_eval")
install_if_needed("matplotlib")
install_if_needed("seaborn")
install_if_needed("pandas")
install_if_needed("numpy")
install_if_needed("tqdm")

print("\n✓ All dependencies installed!")

✓ torch already installed
✓ transformers already installed
✓ datasets already installed
✓ accelerate already installed
✓ auto-gptq already installed


/mnt/miniconda3/envs/quant-env/lib/python3.10/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)


Installing autoawq...
✓ autoawq installed successfully
✓ lm-eval[api] already installed
Installing matplotlib...
✓ matplotlib installed successfully
Installing seaborn...
✓ seaborn installed successfully
✓ pandas already installed
✓ numpy already installed
✓ tqdm already installed

✓ All dependencies installed!


In [12]:
# Evaluation imports
import json
import math
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from collections import defaultdict
from tqdm import tqdm
import time

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [13]:
def evaluate_perplexity(model, tokenizer, dataset_name="wikitext", dataset_config="wikitext-2-raw-v1", 
                        split="test", max_samples=100, max_length=512):
    """
    Evaluate model perplexity on a dataset.
    
    Args:
        model: The language model to evaluate
        tokenizer: The tokenizer
        dataset_name: Name of the dataset
        dataset_config: Configuration of the dataset
        split: Dataset split to use
        max_samples: Maximum number of samples to evaluate
        max_length: Maximum sequence length
    
    Returns:
        Dictionary with perplexity and loss
    """
    print(f"Evaluating perplexity on {dataset_name} {split} split...")
    
    dataset = load_dataset(dataset_name, dataset_config, split=split)
    
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    
    with torch.no_grad():
        for i, item in enumerate(tqdm(dataset, desc="Evaluating", total=min(max_samples, len(dataset)))):
            if i >= max_samples:
                break
            
            text = item["text"].strip()
            if len(text) < 50:  # Skip very short texts
                continue
            
            encodings = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
            input_ids = encodings["input_ids"].to(model.device)
            
            if input_ids.shape[1] < 2:  # Need at least 2 tokens
                continue
            
            outputs = model(input_ids, labels=input_ids)
            loss = outputs.loss
            
            total_loss += loss.item() * input_ids.shape[1]
            total_tokens += input_ids.shape[1]
    
    avg_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
    perplexity = math.exp(avg_loss) if avg_loss < 100 else float('inf')
    
    return {
        "perplexity": perplexity,
        "loss": avg_loss,
        "total_tokens": total_tokens
    }

In [14]:
def evaluate_lm_harness(model_path: str, tasks: List[str] = None, num_fewshot: int = 0, 
                        limit: int = None, device: str = "cuda"):
    """
    Evaluate model using LM Evaluation Harness on downstream tasks.
    
    Args:
        model_path: Path to model directory or HuggingFace model ID
        tasks: List of tasks to evaluate (e.g., ["hellaswag", "arc_easy", "gsm8k"])
        num_fewshot: Number of few-shot examples
        limit: Limit number of samples per task
        device: Device to run on
    
    Returns:
        Dictionary with task results
    """
    try:
        import lm_eval
        from lm_eval.models.huggingface import HFLM
        from lm_eval import simple_evaluate
    except ImportError:
        print("lm-eval not installed. Installing...")
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "lm-eval[api]"])
        import lm_eval
        from lm_eval.models.huggingface import HFLM
        from lm_eval import simple_evaluate
    
    if tasks is None:
        # Default task suite: coding, math, reasoning
        tasks = [
            "hellaswag",      # Commonsense reasoning
            "arc_easy",       # Science questions
            "piqa",           # Physical reasoning
            "winogrande",     # Commonsense reasoning
            "mathqa",         # Math reasoning
            "lambada_openai", # Language understanding
        ]
    
    print(f"Evaluating model on tasks: {tasks}")
    print(f"Model path: {model_path}")
    
    # Initialize model
    lm = HFLM(
        pretrained=model_path,
        device=device,
        dtype="float16",
    )
    
    # Run evaluation
    results = simple_evaluate(
        model=lm,
        tasks=tasks,
        num_fewshot=num_fewshot,
        limit=limit,
        device=device,
    )
    
    return results

In [15]:
def analyze_module_sensitivity(model_id: str = MODEL_ID, bits: int = 4, 
                              eval_samples: int = 50, group_size: int = 128):
    """
    Analyze sensitivity of different module types to quantization.
    Tests quantizing only MLP layers vs only Attention layers vs both.
    
    Args:
        model_id: Model to analyze
        bits: Quantization bits
        eval_samples: Number of samples for perplexity evaluation
        group_size: Group size for quantization
    
    Returns:
        Dictionary with results for each configuration
    """
    print(f"\n{'='*60}")
    print(f"MODULE SENSITIVITY ANALYSIS - {bits}-bit quantization")
    print(f"{'='*60}\n")
    
    tokenizer = load_tokenizer(model_id)
    results = {}
    
    # 1. Baseline - No quantization
    print("1. Evaluating baseline (FP16) model...")
    model = load_base_model(model_id)
    baseline_result = evaluate_perplexity(model, tokenizer, max_samples=eval_samples)
    results["baseline_fp16"] = baseline_result
    print(f"   Baseline Perplexity: {baseline_result['perplexity']:.2f}\n")
    del model
    torch.cuda.empty_cache()
    
    # 2. Quantize only MLP layers
    print(f"2. Evaluating with ONLY MLP layers quantized to {bits}-bit...")
    model = load_base_model(model_id)
    mlp_count = 0
    for name, module in model.named_modules():
        # Look for MLP/FFN layers (common patterns: mlp, feed_forward, fc)
        if any(x in name.lower() for x in ['mlp', 'feed_forward', 'fc1', 'fc2']) and isinstance(module, nn.Linear):
            module.weight.data = _rtn_quantize_tensor(module.weight.data, bits=bits, group_size=group_size)
            mlp_count += 1
    
    mlp_result = evaluate_perplexity(model, tokenizer, max_samples=eval_samples)
    results["mlp_only"] = mlp_result
    results["mlp_only"]["num_layers"] = mlp_count
    print(f"   MLP-only Perplexity: {mlp_result['perplexity']:.2f} ({mlp_count} layers)")
    print(f"   Degradation: {mlp_result['perplexity'] - baseline_result['perplexity']:.2f}\n")
    del model
    torch.cuda.empty_cache()
    
    # 3. Quantize only Attention layers
    print(f"3. Evaluating with ONLY Attention layers quantized to {bits}-bit...")
    model = load_base_model(model_id)
    attn_count = 0
    for name, module in model.named_modules():
        # Look for attention projection layers (q_proj, k_proj, v_proj, o_proj, out_proj)
        if any(x in name.lower() for x in ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'out_proj', 'attn']) \
           and isinstance(module, nn.Linear):
            module.weight.data = _rtn_quantize_tensor(module.weight.data, bits=bits, group_size=group_size)
            attn_count += 1
    
    attn_result = evaluate_perplexity(model, tokenizer, max_samples=eval_samples)
    results["attention_only"] = attn_result
    results["attention_only"]["num_layers"] = attn_count
    print(f"   Attention-only Perplexity: {attn_result['perplexity']:.2f} ({attn_count} layers)")
    print(f"   Degradation: {attn_result['perplexity'] - baseline_result['perplexity']:.2f}\n")
    del model
    torch.cuda.empty_cache()
    
    # 4. Quantize all layers
    print(f"4. Evaluating with ALL layers quantized to {bits}-bit...")
    model = load_base_model(model_id)
    quantize_rtn_inplace(model, bits=bits, group_size=group_size)
    all_result = evaluate_perplexity(model, tokenizer, max_samples=eval_samples)
    results["all_layers"] = all_result
    print(f"   All-layers Perplexity: {all_result['perplexity']:.2f}")
    print(f"   Degradation: {all_result['perplexity'] - baseline_result['perplexity']:.2f}\n")
    del model
    torch.cuda.empty_cache()
    
    # Summary
    print(f"\n{'='*60}")
    print("SENSITIVITY SUMMARY:")
    print(f"{'='*60}")
    print(f"{'Configuration':<20} {'Perplexity':>12} {'Degradation':>12} {'Sensitivity':>12}")
    print(f"{'-'*60}")
    
    baseline_ppl = baseline_result['perplexity']
    for config_name, result in results.items():
        if config_name == "baseline_fp16":
            print(f"{config_name:<20} {result['perplexity']:>12.2f} {'-':>12} {'-':>12}")
        else:
            degradation = result['perplexity'] - baseline_ppl
            sensitivity = (degradation / baseline_ppl) * 100  # Percentage increase
            print(f"{config_name:<20} {result['perplexity']:>12.2f} {degradation:>12.2f} {sensitivity:>11.2f}%")
    
    return results

In [16]:
def calculate_model_size_and_bandwidth(model, bits: int = 16, batch_size: int = 1, 
                                       seq_length: int = 512, include_activations: bool = True):
    """
    Calculate model size and memory bandwidth requirements.
    
    Args:
        model: The model to analyze
        bits: Quantization bits (16 for FP16, 8 for INT8, 4 for INT4)
        batch_size: Batch size for inference
        seq_length: Sequence length
        include_activations: Whether to include activation memory
    
    Returns:
        Dictionary with size and bandwidth metrics
    """
    # Calculate parameter count and memory
    total_params = sum(p.numel() for p in model.parameters())
    
    # Weight memory in bytes
    bytes_per_param = bits / 8
    weight_memory_bytes = total_params * bytes_per_param
    weight_memory_mb = weight_memory_bytes / (1024 ** 2)
    weight_memory_gb = weight_memory_bytes / (1024 ** 3)
    
    # Activation memory (rough estimate for transformer)
    # Activations scale with batch_size * seq_length * hidden_dim * num_layers
    if include_activations:
        # Get model config
        config = model.config
        hidden_size = config.hidden_size
        num_layers = config.num_hidden_layers
        
        # Rough estimate: each layer stores intermediate activations
        # Includes attention scores, MLP intermediates
        activation_memory_bytes = (
            batch_size * seq_length * hidden_size * num_layers * 4 * 2  # FP16 = 2 bytes
        )
        activation_memory_mb = activation_memory_bytes / (1024 ** 2)
    else:
        activation_memory_mb = 0
    
    # Total memory
    total_memory_mb = weight_memory_mb + activation_memory_mb
    total_memory_gb = total_memory_mb / 1024
    
    # Bandwidth estimation (simplified)
    # Assumes each parameter is loaded once per token generation
    # Actual bandwidth depends on hardware and cache efficiency
    bytes_per_token = weight_memory_bytes  # Worst case: load all weights per token
    bandwidth_mb_per_token = bytes_per_token / (1024 ** 2)
    
    return {
        "total_params": total_params,
        "total_params_millions": total_params / 1e6,
        "total_params_billions": total_params / 1e9,
        "weight_memory_mb": weight_memory_mb,
        "weight_memory_gb": weight_memory_gb,
        "activation_memory_mb": activation_memory_mb,
        "total_memory_mb": total_memory_mb,
        "total_memory_gb": total_memory_gb,
        "bits": bits,
        "bandwidth_mb_per_token": bandwidth_mb_per_token,
        "bandwidth_gb_per_token": bandwidth_mb_per_token / 1024,
    }

In [17]:
def comprehensive_evaluation(model_configs: List[dict], 
                            use_lm_harness: bool = True,
                            lm_harness_tasks: List[str] = None,
                            perplexity_samples: int = 100):
    """
    Run comprehensive evaluation on multiple model configurations.
    
    Args:
        model_configs: List of dicts with 'name', 'path', 'bits' keys
        use_lm_harness: Whether to run LM Evaluation Harness
        lm_harness_tasks: Tasks for LM harness
        perplexity_samples: Number of samples for perplexity eval
    
    Returns:
        DataFrame with all results
    """
    results = []
    
    for config in tqdm(model_configs, desc="Evaluating models"):
        print(f"\n{'='*70}")
        print(f"Evaluating: {config['name']}")
        print(f"{'='*70}")
        
        model_path = config['path']
        model_name = config['name']
        bits = config.get('bits', 16)
        
        try:
            # Load model
            print(f"Loading model from {model_path}...")
            if "gptq" in model_name.lower():
                from auto_gptq import AutoGPTQForCausalLM
                model = AutoGPTQForCausalLM.from_quantized(
                    model_path,
                    device="cuda:0",
                    use_safetensors=True,
                )
            elif "awq" in model_name.lower():
                from awq import AutoAWQForCausalLM
                model = AutoAWQForCausalLM.from_quantized(
                    model_path,
                    fuse_layers=True,
                )
            else:
                model = AutoModelForCausalLM.from_pretrained(
                    model_path,
                    torch_dtype=torch.float16,
                    device_map="auto",
                )
            
            tokenizer = AutoTokenizer.from_pretrained(model_path)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            # 1. Calculate size and bandwidth
            print("\n1. Calculating model size and bandwidth...")
            size_metrics = calculate_model_size_and_bandwidth(model, bits=bits)
            
            # 2. Evaluate perplexity
            print("\n2. Evaluating perplexity...")
            ppl_metrics = evaluate_perplexity(model, tokenizer, max_samples=perplexity_samples)
            
            # 3. LM Harness evaluation
            lm_harness_results = {}
            if use_lm_harness:
                print("\n3. Running LM Evaluation Harness...")
                try:
                    harness_results = evaluate_lm_harness(
                        model_path, 
                        tasks=lm_harness_tasks,
                        limit=100  # Limit for faster evaluation
                    )
                    # Extract task scores
                    for task_name, task_result in harness_results.get("results", {}).items():
                        if isinstance(task_result, dict):
                            # Get primary metric (usually 'acc' or 'acc_norm')
                            for metric in ['acc_norm', 'acc', 'exact_match']:
                                if metric in task_result:
                                    lm_harness_results[f"{task_name}_{metric}"] = task_result[metric]
                                    break
                except Exception as e:
                    print(f"   LM Harness evaluation failed: {e}")
                    lm_harness_results = {}
            
            # Compile results
            result = {
                "model_name": model_name,
                "bits": bits,
                "perplexity": ppl_metrics["perplexity"],
                "loss": ppl_metrics["loss"],
                "params_billions": size_metrics["total_params_billions"],
                "weight_memory_gb": size_metrics["weight_memory_gb"],
                "total_memory_gb": size_metrics["total_memory_gb"],
                "bandwidth_gb_per_token": size_metrics["bandwidth_gb_per_token"],
                **lm_harness_results
            }
            results.append(result)
            
            print(f"\n✓ Completed evaluation of {model_name}")
            print(f"  Perplexity: {ppl_metrics['perplexity']:.2f}")
            print(f"  Model size: {size_metrics['weight_memory_gb']:.2f} GB")
            print(f"  Bandwidth: {size_metrics['bandwidth_gb_per_token']:.3f} GB/token")
            
        except Exception as e:
            print(f"\n✗ Failed to evaluate {model_name}: {e}")
            results.append({
                "model_name": model_name,
                "bits": bits,
                "perplexity": None,
                "error": str(e)
            })
        
        finally:
            # Cleanup
            if 'model' in locals():
                del model
            torch.cuda.empty_cache()
    
    return pd.DataFrame(results)

In [18]:
def plot_tradeoff_analysis(results_df: pd.DataFrame, output_dir: Path = None):
    """
    Create comprehensive trade-off visualizations.
    
    Args:
        results_df: DataFrame from comprehensive_evaluation
        output_dir: Directory to save plots
    """
    if output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
    
    # Filter valid results
    df = results_df[results_df['perplexity'].notna()].copy()
    
    if len(df) == 0:
        print("No valid results to plot!")
        return
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Quantization Trade-off Analysis', fontsize=16, fontweight='bold')
    
    # 1. Perplexity vs Memory Bandwidth (Main Trade-off)
    ax1 = axes[0, 0]
    scatter = ax1.scatter(df['bandwidth_gb_per_token'], df['perplexity'], 
                s=df['weight_memory_gb']*50, alpha=0.6, c=df['bits'], cmap='viridis')
    
    for idx, row in df.iterrows():
        ax1.annotate(row['model_name'], 
                    (row['bandwidth_gb_per_token'], row['perplexity']),
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    ax1.set_xlabel('Memory Bandwidth (GB/token)', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Perplexity (lower is better)', fontsize=11, fontweight='bold')
    ax1.set_title('Accuracy vs Memory Bandwidth\n(size = model memory)', fontsize=12)
    ax1.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax1, label='Quantization Bits')
    
    # Find and highlight the "sweet spot" (best perplexity/bandwidth ratio)
    df['efficiency'] = df['perplexity'] / df['bandwidth_gb_per_token']
    best_idx = df['efficiency'].idxmin()
    best_row = df.loc[best_idx]
    ax1.scatter(best_row['bandwidth_gb_per_token'], best_row['perplexity'], 
               s=300, marker='*', c='red', edgecolors='darkred', linewidths=2, 
               label='Sweet Spot', zorder=5)
    ax1.legend()
    
    # 2. Model Size vs Perplexity
    ax2 = axes[0, 1]
    colors_bits = {16: 'blue', 8: 'green', 4: 'orange', 2: 'red'}
    for bits in sorted(df['bits'].unique()):
        subset = df[df['bits'] == bits]
        ax2.scatter(subset['weight_memory_gb'], subset['perplexity'], 
                   label=f'{bits}-bit', s=100, alpha=0.7, color=colors_bits.get(bits, 'gray'))
    
    ax2.set_xlabel('Model Size (GB)', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Perplexity', fontsize=11, fontweight='bold')
    ax2.set_title('Model Size vs Perplexity', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Bits vs Perplexity (grouped by method)
    ax3 = axes[1, 0]
    
    # Extract method from model name
    df['method'] = df['model_name'].apply(lambda x: x.split('_')[0] if '_' in x else 'base')
    
    methods = df['method'].unique()
    x_positions = np.arange(len(df))
    bar_width = 0.35
    
    bars = ax3.bar(x_positions, df['perplexity'], color=df['bits'].map(colors_bits), alpha=0.7)
    ax3.set_xlabel('Model Configuration', fontsize=11, fontweight='bold')
    ax3.set_ylabel('Perplexity', fontsize=11, fontweight='bold')
    ax3.set_title('Perplexity by Configuration', fontsize=12)
    ax3.set_xticks(x_positions)
    ax3.set_xticklabels(df['model_name'], rotation=45, ha='right', fontsize=8)
    ax3.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        if not np.isnan(height):
            ax3.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1f}', ha='center', va='bottom', fontsize=8)
    
    # 4. Summary Table
    ax4 = axes[1, 1]
    ax4.axis('tight')
    ax4.axis('off')
    
    # Create summary table
    summary_data = []
    for idx, row in df.iterrows():
        summary_data.append([
            row['model_name'][:20],  # Truncate long names
            f"{row['bits']}",
            f"{row['perplexity']:.2f}",
            f"{row['weight_memory_gb']:.2f}",
            f"{row['bandwidth_gb_per_token']:.3f}"
        ])
    
    table = ax4.table(cellText=summary_data,
                     colLabels=['Model', 'Bits', 'Perplexity', 'Size (GB)', 'BW (GB/tok)'],
                     cellLoc='center',
                     loc='center',
                     bbox=[0, 0, 1, 1])
    
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)
    
    # Style header
    for i in range(5):
        table[(0, i)].set_facecolor('#4CAF50')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Alternate row colors
    for i in range(1, len(summary_data) + 1):
        color = '#f0f0f0' if i % 2 == 0 else 'white'
        for j in range(5):
            table[(i, j)].set_facecolor(color)
    
    ax4.set_title('Performance Summary', fontsize=12, pad=20)
    
    plt.tight_layout()
    
    if output_dir:
        plt.savefig(output_dir / 'tradeoff_analysis.png', dpi=300, bbox_inches='tight')
        print(f"Saved plot to {output_dir / 'tradeoff_analysis.png'}")
    
    plt.show()
    
    # Print recommendations
    print("\n" + "="*70)
    print("RECOMMENDATIONS FOR MOBILE DEPLOYMENT")
    print("="*70)
    
    print(f"\n🌟 SWEET SPOT MODEL: {best_row['model_name']}")
    print(f"   - Quantization: {best_row['bits']}-bit")
    print(f"   - Perplexity: {best_row['perplexity']:.2f}")
    print(f"   - Model Size: {best_row['weight_memory_gb']:.2f} GB")
    print(f"   - Bandwidth: {best_row['bandwidth_gb_per_token']:.3f} GB/token")
    print(f"   - Efficiency Score: {best_row['efficiency']:.2f}")
    
    # Find best for different constraints
    print("\n📱 For Mobile Devices (< 2GB):")
    mobile_df = df[df['weight_memory_gb'] < 2]
    if len(mobile_df) > 0:
        best_mobile = mobile_df.loc[mobile_df['perplexity'].idxmin()]
        print(f"   Recommended: {best_mobile['model_name']}")
        print(f"   Size: {best_mobile['weight_memory_gb']:.2f} GB, Perplexity: {best_mobile['perplexity']:.2f}")
    else:
        print("   No models under 2GB available")
    
    print("\n⚡ For Best Accuracy (accept higher memory):")
    best_acc = df.loc[df['perplexity'].idxmin()]
    print(f"   Recommended: {best_acc['model_name']}")
    print(f"   Size: {best_acc['weight_memory_gb']:.2f} GB, Perplexity: {best_acc['perplexity']:.2f}")
    
    print("\n" + "="*70)

In [19]:
def plot_module_sensitivity(sensitivity_results: dict, output_dir: Path = None):
    """
    Visualize module sensitivity analysis results.
    
    Args:
        sensitivity_results: Results from analyze_module_sensitivity()
        output_dir: Directory to save plots
    """
    if output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
    
    # Extract data
    configs = []
    perplexities = []
    degradations = []
    
    baseline_ppl = sensitivity_results['baseline_fp16']['perplexity']
    
    for config_name, result in sensitivity_results.items():
        if config_name == 'baseline_fp16':
            configs.append('Baseline\n(FP16)')
            perplexities.append(result['perplexity'])
            degradations.append(0)
        else:
            display_name = config_name.replace('_', '\n').title()
            configs.append(display_name)
            perplexities.append(result['perplexity'])
            degradations.append(result['perplexity'] - baseline_ppl)
    
    # Create subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Module Sensitivity to Quantization', fontsize=14, fontweight='bold')
    
    # Plot 1: Perplexity by configuration
    colors = ['green', 'orange', 'red', 'darkred']
    bars1 = ax1.bar(configs, perplexities, color=colors, alpha=0.7, edgecolor='black')
    ax1.set_ylabel('Perplexity', fontsize=11, fontweight='bold')
    ax1.set_xlabel('Configuration', fontsize=11, fontweight='bold')
    ax1.set_title('Perplexity by Quantization Configuration', fontsize=12)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}', ha='center', va='bottom', fontsize=10)
    
    # Plot 2: Degradation (sensitivity)
    bars2 = ax2.bar(configs[1:], degradations[1:], color=colors[1:], alpha=0.7, edgecolor='black')
    ax2.set_ylabel('Perplexity Degradation', fontsize=11, fontweight='bold')
    ax2.set_xlabel('Configuration', fontsize=11, fontweight='bold')
    ax2.set_title('Sensitivity to Quantization (vs Baseline)', fontsize=12)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels and percentage
    for i, bar in enumerate(bars2):
        height = bar.get_height()
        pct = (height / baseline_ppl) * 100
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    
    if output_dir:
        plt.savefig(output_dir / 'module_sensitivity.png', dpi=300, bbox_inches='tight')
        print(f"Saved plot to {output_dir / 'module_sensitivity.png'}")
    
    plt.show()
    
    # Print analysis
    print("\n" + "="*70)
    print("MODULE SENSITIVITY INSIGHTS")
    print("="*70)
    
    if degradations[1] < degradations[2]:  # MLP less sensitive than Attention
        print("\n🔍 Finding: MLP layers are MORE ROBUST to quantization than Attention layers")
        print(f"   - MLP degradation: {degradations[1]:.2f} ({(degradations[1]/baseline_ppl)*100:.1f}%)")
        print(f"   - Attention degradation: {degradations[2]:.2f} ({(degradations[2]/baseline_ppl)*100:.1f}%)")
        print("\n💡 Recommendation: Consider mixed-precision quantization:")
        print("   - Quantize MLP layers more aggressively (4-bit)")
        print("   - Keep Attention layers at higher precision (8-bit or FP16)")
    else:
        print("\n🔍 Finding: Attention layers are MORE ROBUST to quantization than MLP layers")
        print(f"   - Attention degradation: {degradations[2]:.2f} ({(degradations[2]/baseline_ppl)*100:.1f}%)")
        print(f"   - MLP degradation: {degradations[1]:.2f} ({(degradations[1]/baseline_ppl)*100:.1f}%)")
        print("\n💡 Recommendation: Consider mixed-precision quantization:")
        print("   - Quantize Attention layers more aggressively (4-bit)")
        print("   - Keep MLP layers at higher precision (8-bit or FP16)")
    
    print("\n" + "="*70)

In [20]:
# Example 1: Module Sensitivity Analysis
# This analyzes which layers (MLP vs Attention) are most sensitive to quantization

print("Running module sensitivity analysis...")
sensitivity_results = analyze_module_sensitivity(
    model_id=MODEL_ID,
    bits=4,  # Test with 4-bit quantization
    eval_samples=50,  # Use fewer samples for faster evaluation
    group_size=128
)

# Visualize results
plot_module_sensitivity(sensitivity_results, output_dir=OUTPUT_ROOT / "analysis")

# Save results to JSON
with open(OUTPUT_ROOT / "analysis" / "module_sensitivity.json", "w") as f:
    json.dump({k: {kk: vv for kk, vv in v.items() if kk != 'total_tokens'} 
               for k, v in sensitivity_results.items()}, f, indent=2)

print("\n✓ Module sensitivity analysis complete!")

Running module sensitivity analysis...

MODULE SENSITIVITY ANALYSIS - 4-bit quantization



/mnt/miniconda3/envs/quant-env/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Using pad_token, but it is not set yet.


1. Evaluating baseline (FP16) model...


OSError:  model.safetensors or model.safetensors.index.json and thus cannot be loaded with `safetensors`. Please make sure that the model has been saved with `safe_serialization=True` or do not set `use_safetensors=True`.

In [ ]:
# Example 2: Comprehensive Evaluation of All Quantized Models
# This evaluates all quantized models and compares them

# Define model configurations to evaluate
model_configs = [
    {
        "name": "baseline_fp16",
        "path": MODEL_ID,
        "bits": 16
    },
    {
        "name": "rtn_w8",
        "path": str(OUTPUT_ROOT / "rtn_w8"),
        "bits": 8
    },
    {
        "name": "rtn_w4",
        "path": str(OUTPUT_ROOT / "rtn_w4"),
        "bits": 4
    },
    {
        "name": "gptq_w4",
        "path": str(OUTPUT_ROOT / "gptq_w4"),
        "bits": 4
    },
    {
        "name": "awq_w4",
        "path": str(OUTPUT_ROOT / "awq_w4"),
        "bits": 4
    },
]

# Run comprehensive evaluation
print("Running comprehensive evaluation on all models...")
print("This may take a while...\n")

results_df = comprehensive_evaluation(
    model_configs=model_configs,
    use_lm_harness=True,  # Set to False to skip LM harness and run faster
    lm_harness_tasks=[
        "hellaswag",      # Commonsense reasoning
        "arc_easy",       # Science QA
        "winogrande",     # Commonsense
    ],
    perplexity_samples=100
)

# Display results
print("\n" + "="*70)
print("EVALUATION RESULTS")
print("="*70)
display(results_df)

# Save results
results_df.to_csv(OUTPUT_ROOT / "analysis" / "evaluation_results.csv", index=False)
print(f"\n✓ Results saved to {OUTPUT_ROOT / 'analysis' / 'evaluation_results.csv'}")

# Create visualizations
plot_tradeoff_analysis(results_df, output_dir=OUTPUT_ROOT / "analysis")

print("\n✓ Comprehensive evaluation complete!")

In [ ]:
# Example 3: Quick Single Model Evaluation
# Evaluate just one model quickly for testing

# Choose a model to evaluate
test_model_path = OUTPUT_ROOT / "rtn_w4"
test_model_bits = 4

if test_model_path.exists():
    print(f"Quick evaluation of {test_model_path}...")
    
    # Load model and tokenizer
    model = AutoModelForCausalLM.from_pretrained(
        test_model_path,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    tokenizer = load_tokenizer(test_model_path)
    
    # Calculate metrics
    print("\n1. Model size and bandwidth:")
    size_info = calculate_model_size_and_bandwidth(model, bits=test_model_bits)
    print(f"   Parameters: {size_info['total_params_billions']:.2f}B")
    print(f"   Weight memory: {size_info['weight_memory_gb']:.2f} GB")
    print(f"   Bandwidth per token: {size_info['bandwidth_gb_per_token']:.3f} GB")
    
    # Evaluate perplexity
    print("\n2. Perplexity:")
    ppl_result = evaluate_perplexity(model, tokenizer, max_samples=20)
    print(f"   Perplexity: {ppl_result['perplexity']:.2f}")
    print(f"   Loss: {ppl_result['loss']:.4f}")
    
    # Generate sample text
    print("\n3. Sample generation:")
    prompt = "The future of artificial intelligence is"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_length=50, do_sample=True, temperature=0.7)
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"   Prompt: {prompt}")
    print(f"   Generated: {generated}")
    
    del model
    torch.cuda.empty_cache()
    print("\n✓ Quick evaluation complete!")
else:
    print(f"Model not found at {test_model_path}. Please run quantization first.")

## Example 4: Complete Evaluation Pipeline with Visualizations

This is a comprehensive example that runs the full evaluation pipeline and generates all visualizations:
1. Module sensitivity analysis with plots
2. Comprehensive evaluation of all models
3. Trade-off analysis visualizations
4. Deployment recommendations

In [ ]:
# ==============================================================================
# COMPLETE EVALUATION PIPELINE - Run this after quantization is complete
# ==============================================================================

import os
os.makedirs(OUTPUT_ROOT / "analysis", exist_ok=True)

print("="*80)
print("COMPLETE EVALUATION PIPELINE")
print("="*80)
print("\nThis will:")
print("1. Analyze module sensitivity (MLP vs Attention)")
print("2. Evaluate all quantized models")
print("3. Generate comprehensive visualizations")
print("4. Provide deployment recommendations")
print("\n" + "="*80 + "\n")

# ------------------------------------------------------------------------------
# STEP 1: Module Sensitivity Analysis
# ------------------------------------------------------------------------------
print("\n📊 STEP 1/3: Module Sensitivity Analysis")
print("-" * 80)
print("Analyzing which layers (MLP vs Attention) are most sensitive to quantization...")

try:
    sensitivity_results = analyze_module_sensitivity(
        model_id=MODEL_ID,
        bits=4,  # Test with 4-bit quantization
        eval_samples=30,  # Use fewer samples for faster demo (increase to 100+ for production)
        group_size=128
    )
    
    # Generate and save visualization
    print("\n📈 Generating module sensitivity plots...")
    plot_module_sensitivity(sensitivity_results, output_dir=OUTPUT_ROOT / "analysis")
    
    # Save raw results
    import json
    with open(OUTPUT_ROOT / "analysis" / "module_sensitivity_results.json", "w") as f:
        # Remove non-serializable fields for JSON
        clean_results = {
            k: {kk: vv for kk, vv in v.items() if kk != 'total_tokens'} 
            for k, v in sensitivity_results.items()
        }
        json.dump(clean_results, f, indent=2)
    
    print(f"✓ Sensitivity analysis complete! Results saved to {OUTPUT_ROOT / 'analysis'}")
    
except Exception as e:
    print(f"⚠️  Module sensitivity analysis failed: {e}")
    print("   Continuing with other evaluations...")
    sensitivity_results = None

# ------------------------------------------------------------------------------
# STEP 2: Comprehensive Model Evaluation
# ------------------------------------------------------------------------------
print("\n\n📊 STEP 2/3: Comprehensive Model Evaluation")
print("-" * 80)
print("Evaluating all quantized models (perplexity + LM harness tasks)...")
print("Note: This may take 15-30 minutes depending on hardware\n")

# Define which models to evaluate
model_configs = []

# Add baseline if needed
if True:  # Set to True to evaluate baseline FP16 model
    model_configs.append({
        "name": "baseline_fp16",
        "path": MODEL_ID,
        "bits": 16
    })

# Add quantized models (only those that exist)
for model_name, bits in [
    ("rtn_w8", 8),
    ("rtn_w4", 4),
    ("gptq_w4", 4),
    ("awq_w4", 4),
]:
    model_path = OUTPUT_ROOT / model_name
    if model_path.exists():
        model_configs.append({
            "name": model_name,
            "path": str(model_path),
            "bits": bits
        })
        print(f"  ✓ Found: {model_name}")
    else:
        print(f"  ⚠️  Skipping: {model_name} (not found)")

if not model_configs:
    print("\n⚠️  No quantized models found! Please run quantization first.")
    print("   Run the quantization cells before evaluation.")
else:
    print(f"\n📋 Will evaluate {len(model_configs)} model(s)\n")
    
    try:
        # Run comprehensive evaluation
        results_df = comprehensive_evaluation(
            model_configs=model_configs,
            use_lm_harness=True,  # Set to False for faster evaluation (skip downstream tasks)
            lm_harness_tasks=[
                "hellaswag",      # Commonsense reasoning
                "arc_easy",       # Science questions  
                "winogrande",     # Commonsense reasoning
            ],
            perplexity_samples=50  # Increase to 100+ for production
        )
        
        # Display results
        print("\n" + "="*80)
        print("EVALUATION RESULTS SUMMARY")
        print("="*80)
        print(results_df.to_string())
        
        # Save results
        results_df.to_csv(OUTPUT_ROOT / "analysis" / "evaluation_results.csv", index=False)
        results_df.to_json(OUTPUT_ROOT / "analysis" / "evaluation_results.json", orient="records", indent=2)
        print(f"\n✓ Results saved to {OUTPUT_ROOT / 'analysis' / 'evaluation_results.csv'}")
        
    except Exception as e:
        print(f"⚠️  Comprehensive evaluation failed: {e}")
        print("   You can still view sensitivity analysis results.")
        results_df = None

# ------------------------------------------------------------------------------
# STEP 3: Trade-off Analysis and Visualizations
# ------------------------------------------------------------------------------
print("\n\n📊 STEP 3/3: Trade-off Analysis & Visualizations")
print("-" * 80)

if results_df is not None and len(results_df) > 0:
    print("Generating comprehensive trade-off visualizations...")
    
    try:
        # Generate all visualizations
        plot_tradeoff_analysis(results_df, output_dir=OUTPUT_ROOT / "analysis")
        
        print(f"\n✓ All visualizations generated and saved to {OUTPUT_ROOT / 'analysis'}")
        
    except Exception as e:
        print(f"⚠️  Visualization generation failed: {e}")
else:
    print("⚠️  Skipping visualizations (no evaluation results available)")

# ------------------------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------------------------
print("\n\n" + "="*80)
print(" EVALUATION PIPELINE COMPLETE!")
print("="*80)

print("\n📁 Output Files:")
print(f"   {OUTPUT_ROOT / 'analysis' / 'module_sensitivity.png'}")
print(f"   {OUTPUT_ROOT / 'analysis' / 'module_sensitivity_results.json'}")
print(f"   {OUTPUT_ROOT / 'analysis' / 'tradeoff_analysis.png'}")
print(f"   {OUTPUT_ROOT / 'analysis' / 'evaluation_results.csv'}")
print(f"   {OUTPUT_ROOT / 'analysis' / 'evaluation_results.json'}")

print("\n📊 Key Findings:")
if sensitivity_results:
    baseline_ppl = sensitivity_results['baseline_fp16']['perplexity']
    mlp_ppl = sensitivity_results['mlp_only']['perplexity']
    attn_ppl = sensitivity_results['attention_only']['perplexity']
    
    mlp_deg = mlp_ppl - baseline_ppl
    attn_deg = attn_ppl - baseline_ppl
    
    if mlp_deg < attn_deg:
        print(f"   • MLP layers are MORE robust to quantization (degradation: {mlp_deg:.2f})")
        print(f"   • Attention layers are MORE sensitive (degradation: {attn_deg:.2f})")
        print("   • Recommendation: Use mixed precision - quantize MLP more aggressively")
    else:
        print(f"   • Attention layers are MORE robust to quantization (degradation: {attn_deg:.2f})")
        print(f"   • MLP layers are MORE sensitive (degradation: {mlp_deg:.2f})")
        print("   • Recommendation: Use mixed precision - quantize Attention more aggressively")

if results_df is not None and len(results_df) > 0:
    best_model = results_df.loc[results_df['perplexity'].idxmin()]
    print(f"\n   • Best model: {best_model['model_name']}")
    print(f"     - Perplexity: {best_model['perplexity']:.2f}")
    print(f"     - Size: {best_model['weight_memory_gb']:.2f} GB")
    print(f"     - Bandwidth: {best_model['bandwidth_gb_per_token']:.3f} GB/token")

print("\n" + "="*80)
print("✓ Check the plots and CSV files for detailed analysis!")
print("="*80 + "\n")

## Example 5: Re-plot from Saved Results

If you've already run the evaluation and want to regenerate plots with different settings, use this cell to load the saved results and re-plot.

In [ ]:
# ==============================================================================
# RE-PLOT FROM SAVED RESULTS
# ==============================================================================

import pandas as pd
import json

print("Loading saved evaluation results...")

# Load evaluation results
results_csv_path = OUTPUT_ROOT / "analysis" / "evaluation_results.csv"
sensitivity_json_path = OUTPUT_ROOT / "analysis" / "module_sensitivity_results.json"

# Check if files exist
if results_csv_path.exists():
    print(f"✓ Found evaluation results: {results_csv_path}")
    results_df = pd.read_csv(results_csv_path)
    
    print("\nLoaded results:")
    print(results_df[['model_name', 'bits', 'perplexity', 'weight_memory_gb']].to_string())
    
    # Re-generate trade-off plots
    print("\n📈 Generating trade-off analysis plots...")
    plot_tradeoff_analysis(results_df, output_dir=OUTPUT_ROOT / "analysis")
    print(f"✓ Trade-off plots saved to {OUTPUT_ROOT / 'analysis' / 'tradeoff_analysis.png'}")
    
else:
    print(f"⚠️  No evaluation results found at {results_csv_path}")
    print("   Run Example 4 first to generate results.")

# Load and plot module sensitivity results
if sensitivity_json_path.exists():
    print(f"\n✓ Found sensitivity results: {sensitivity_json_path}")
    
    with open(sensitivity_json_path, 'r') as f:
        sensitivity_results = json.load(f)
    
    # Re-generate sensitivity plots
    print("\n📈 Generating module sensitivity plots...")
    plot_module_sensitivity(sensitivity_results, output_dir=OUTPUT_ROOT / "analysis")
    print(f"✓ Sensitivity plots saved to {OUTPUT_ROOT / 'analysis' / 'module_sensitivity.png'}")
    
else:
    print(f"\n⚠️  No sensitivity results found at {sensitivity_json_path}")
    print("   Run Example 1 or Example 4 first to generate results.")

print("\n✓ Re-plotting complete!")

## Example: Run Complete Evaluation Pipeline

Below are examples of how to use the evaluation functions.

## Summary: Evaluation Functions Available

### 📊 Evaluation Functions

1. **`evaluate_perplexity(model, tokenizer, ...)`**
   - Evaluates model perplexity on WikiText-2 test set
   - Returns perplexity score and loss

2. **`evaluate_lm_harness(model_path, tasks, ...)`**
   - Uses LM Evaluation Harness for downstream tasks
   - Supports tasks: hellaswag, arc_easy, winogrande, mathqa, lambada, etc.
   - Evaluates coding, math, and reasoning capabilities

3. **`analyze_module_sensitivity(model_id, bits, ...)`**
   - Compares quantization impact on different module types
   - Tests: Baseline, MLP-only, Attention-only, All layers
   - Identifies which layers are most sensitive to quantization noise

4. **`calculate_model_size_and_bandwidth(model, bits, ...)`**
   - Calculates model memory footprint
   - Estimates memory bandwidth requirements
   - Useful for deployment planning

5. **`comprehensive_evaluation(model_configs, ...)`**
   - Runs complete evaluation pipeline on multiple models
   - Combines perplexity, LM harness, and memory metrics
   - Returns DataFrame with all results

### 📈 Visualization Functions

1. **`plot_tradeoff_analysis(results_df, ...)`**
   - Creates 4-panel visualization:
     - Accuracy vs Memory Bandwidth (with sweet spot)
     - Model Size vs Perplexity
     - Perplexity by Configuration
     - Summary Table
   - Provides deployment recommendations

2. **`plot_module_sensitivity(sensitivity_results, ...)`**
   - Visualizes module-specific quantization impact
   - Shows which layers (MLP vs Attention) are most sensitive
   - Provides mixed-precision quantization recommendations

### 🎯 Key Insights Provided

- **Sweet Spot Identification**: Best accuracy/bandwidth trade-off
- **Module Sensitivity**: Which layers tolerate quantization better
- **Deployment Recommendations**: Best model for mobile (<2GB) vs accuracy
- **Memory Bandwidth Analysis**: Critical for mobile deployment
- **Downstream Task Performance**: Beyond just perplexity

### 📝 Typical Workflow

**Quick Start:**
- Run **Example 4** (Complete Evaluation Pipeline) - does everything automatically!

**Step-by-step:**
1. Run quantization cells (Sections 1-7) to generate quantized models
2. Run **Example 1** for module sensitivity analysis (MLP vs Attention)
3. Run **Example 2** for comprehensive evaluation of all models
4. Visualizations are automatically generated and saved

**Re-plotting:**
- Run **Example 5** to regenerate plots from saved results (useful for tweaking visualizations)

**Quick Testing:**
- Run **Example 3** to quickly test a single model without full evaluation